# SORTING CODE
## CODE SEQUENCE 1
### INPUT: cleaned responses from the 2-back-task
### OUTPUT: resposnes ready to be worked in the next steps in csv format
This code take the results obtained and make it a csv file to be workable.


In [1]:
from pathlib import Path
import json
import re
import pandas as pd


# ============================================================
# CONFIGURAZIONE
# ============================================================

BASE_PATH = Path(
    r"C:\Users\a.genua\OneDrive - Scuola Superiore Sant'Anna\PROGETTI\Progetti in corso\Articoli in corso\RA-L_&_Sanseverino\FlightSimulator"
)

EXCEL_NAME = "2-BACK-TASK-analisi_V2.xlsx"
OUTPUT_CSV_NAME = "2back_human_factors_export_final.csv"

# mapping colori dal codice Unity
COLOR_MAP = {
    0: "Green",
    1: "Red",
    2: "Yellow",
    3: "Blue",
    4: "Black",
}

# normalizzazione risposte Excel
RESPONSE_MAP = {
    "ROSSO": "Red",
    "RED": "Red",
    "BLU": "Blue",
    "BLUE": "Blue",
    "VERDE": "Green",
    "GREEN": "Green",
    "GIALLO": "Yellow",
    "YELLOW": "Yellow",
    "NERO": "Black",
    "BLACK": "Black",
    "FAIL": "",
    "": "",
}


# ============================================================
# UTILS
# ============================================================

def subject_sort_key(name: str):
    m = re.search(r"(\d+)", name)
    return int(m.group(1)) if m else 9999


def subject_to_id(subject_name: str) -> str:
    """
    SUB1 -> ID1
    SUB10 -> ID10
    """
    m = re.search(r"(\d+)", subject_name)
    if not m:
        return subject_name
    return f"ID{int(m.group(1))}"


def load_multiple_json_objects(file_path: Path):
    """
    Il file main .txt contiene molti oggetti JSON in sequenza.
    """
    with open(file_path, "r", encoding="utf-8") as f:
        content = f.read().strip()

    decoder = json.JSONDecoder()
    idx = 0
    objects = []

    while idx < len(content):
        while idx < len(content) and content[idx].isspace():
            idx += 1

        if idx >= len(content):
            break

        obj, end = decoder.raw_decode(content, idx)
        objects.append(obj)
        idx = end

    return objects


def normalize_response(value) -> str:
    if pd.isna(value):
        return ""
    s = str(value).strip().upper()
    return RESPONSE_MAP.get(s, s.title())


# ============================================================
# SCANSIONE FILE
# ============================================================

def collect_condition_files(base_path: Path):
    """
    Ritorna un dict:
    files[(subject, condition)] = {
        "main_file": Path(...),
        "target_file": Path(...)
    }
    """
    files = {}

    subfolders = sorted(
        [p for p in base_path.iterdir() if p.is_dir() and p.name.upper().startswith("SUB")],
        key=lambda p: subject_sort_key(p.name)
    )

    for sub_folder in subfolders:
        subject = sub_folder.name

        for cond_folder in sub_folder.iterdir():
            if not cond_folder.is_dir():
                continue

            folder_name = cond_folder.name.upper()

            if "NOMINAL_POSITION" in folder_name:
                condition = "NOMINAL"
            elif "LOW_MIS" in folder_name:
                condition = "LOW"
            elif "HIGH_MIS" in folder_name:
                condition = "HIGH"
            else:
                continue

            target_files = list(cond_folder.glob("*_targets.txt"))
            if not target_files:
                continue

            # assumiamo uno solo
            target_file = target_files[0]
            main_file = target_file.with_name(target_file.name.replace("_targets.txt", ".txt"))

            if not main_file.exists():
                print(f"[WARNING] Main file mancante per {subject} {condition}: {target_file.name}")
                continue

            files[(subject, condition)] = {
                "main_file": main_file,
                "target_file": target_file,
            }

    return files


# ============================================================
# ESTRAZIONE COLORI
# ============================================================

def extract_target_colors_from_targets(targets_data: dict):
    """
    Colonna 1: colori così come sono nel file _targets.txt
    """
    targets_list = targets_data.get("Targets", [])
    return [COLOR_MAP.get(t.get("Color"), f"UNK_{t.get('Color')}") for t in targets_list]


def extract_log_colors_from_main(main_data_list, targets_data: dict):
    """
    Colonna 2: colori ricostruiti dal file log principale.
    Prendiamo gli eventi come cambi di TargetID.
    """
    targets_list = targets_data.get("Targets", [])
    log_colors = []

    prev_target_id = None
    for sample in main_data_list:
        target_id = sample.get("TargetID")

        if target_id != prev_target_id:
            if isinstance(target_id, int) and 0 <= target_id < len(targets_list):
                color_id = targets_list[target_id].get("Color")
                color_name = COLOR_MAP.get(color_id, f"UNK_{color_id}")
                log_colors.append(color_name)
            else:
                log_colors.append("")

            prev_target_id = target_id

    return log_colors


def compute_expected_2back(log_colors):
    """
    Colonna 3: solo risposte corrette attese.
    Si compila SOLO quando il target corrente è Black.
    """
    expected = []
    for i, c in enumerate(log_colors):
        if c == "Black" and i >= 2:
            expected.append(log_colors[i - 2])
    return expected


# ============================================================
# LETTURA EXCEL
# ============================================================

def load_excel_responses(excel_path: Path):
    df_excel = pd.read_excel(excel_path, header=None)

    # Riga 0: soggetti (con celle merge)
    # Riga 1: condizioni
    header_subjects = df_excel.iloc[0, :].ffill()
    header_conditions = df_excel.iloc[1, :]
    data_rows = df_excel.iloc[2:, :]

    responses = {}

    print("\n=== RISPOSTE TROVATE NELL'EXCEL ===")

    for col_idx in range(df_excel.shape[1]):
        subject_cell = str(header_subjects[col_idx]).strip()
        condition_cell = str(header_conditions[col_idx]).strip().upper()

        # salta colonna iniziale "SUBJECT" o colonne non valide
        if subject_cell.upper() == "SUBJECT" or subject_cell == "nan":
            continue

        match = re.search(r"(\d+)", subject_cell)
        if not match:
            continue

        subject = f"SUB{int(match.group(1))}"

        if "NOMINAL" in condition_cell:
            condition = "NOMINAL"
        elif "LOW" in condition_cell:
            condition = "LOW"
        elif "HIGH" in condition_cell:
            condition = "HIGH"
        else:
            continue

        col_data = data_rows.iloc[:, col_idx].tolist()
        normalized = [normalize_response(v) for v in col_data]
        normalized = [v for v in normalized if v != ""]

        responses[(subject, condition)] = normalized

    for k in sorted(responses.keys()):
        print(k, len(responses[k]), responses[k][:5])

    return responses


# ============================================================
# COSTRUZIONE BLOCCO 4 COLONNE
# ============================================================

def build_block_dataframe(id_label, condition, target_colors, log_colors, expected_responses, subject_responses):
    """
    Costruisce il blocco 4 colonne per una coppia soggetto-condizione.
    Le liste possono avere lunghezze diverse: vengono paddate con stringa vuota.
    """
    n_rows = max(
        len(target_colors),
        len(log_colors),
        len(expected_responses),
        len(subject_responses),
        1
    )

    def pad(lst):
        return lst + [""] * (n_rows - len(lst))

    block = pd.DataFrame({
        f"{id_label}_{condition}_targets_color": pad(target_colors),
        f"{id_label}_{condition}_log_color": pad(log_colors),
        f"{id_label}_{condition}_expected_response": pad(expected_responses),
        f"{id_label}_{condition}_subject_response": pad(subject_responses),
    })

    return block


# ============================================================
# PIPELINE PRINCIPALE
# ============================================================

def main():
    print(f"[INFO] Base path: {BASE_PATH}")

    files = collect_condition_files(BASE_PATH)
    if not files:
        raise RuntimeError("Nessun file valido trovato nelle cartelle SUB*.")

    subjects = sorted({k[0] for k in files.keys()}, key=subject_sort_key)
    print(f"[INFO] Soggetti trovati: {subjects}")

    excel_path = BASE_PATH / EXCEL_NAME
    responses = load_excel_responses(excel_path)
    print("=== RISPOSTE TROVATE NELL'EXCEL ===")
    for k in sorted(responses.keys()):
        print(k, len(responses[k]), responses[k][:5])

    
    condition_order = ["NOMINAL", "LOW", "HIGH"]
    all_blocks = []

    for subject in subjects:
        id_label = subject_to_id(subject)

        for condition in condition_order:
            key = (subject, condition)

            if key not in files:
                print(f"[WARNING] Nessun file per {subject} - {condition}")
                # creo comunque il blocco vuoto
                empty_block = build_block_dataframe(
                    id_label=id_label,
                    condition=condition,
                    target_colors=[],
                    log_colors=[],
                    expected_responses=[],
                    subject_responses=responses.get(key, [])
                )
                all_blocks.append(empty_block)
                continue

            main_file = files[key]["main_file"]
            target_file = files[key]["target_file"]

            print(f"[INFO] Elaboro {subject} - {condition}")

            # carica target file
            with open(target_file, "r", encoding="utf-8") as f:
                targets_data = json.load(f)

            # carica main file
            main_data_list = load_multiple_json_objects(main_file)

            # 1) colori dal file targets
            target_colors = extract_target_colors_from_targets(targets_data)

            # 2) colori ricostruiti dal log principale
            log_colors = extract_log_colors_from_main(main_data_list, targets_data)

            # 3) risposte corrette attese (solo ai black)
            expected_responses = compute_expected_2back(log_colors)

            # 4) risposte reali dal soggetto
            subject_responses = responses.get(key, [])

            block_df = build_block_dataframe(
                id_label=id_label,
                condition=condition,
                target_colors=target_colors,
                log_colors=log_colors,
                expected_responses=expected_responses,
                subject_responses=subject_responses,
            )

            all_blocks.append(block_df)

    final_df = pd.concat(all_blocks, axis=1)

    output_csv = BASE_PATH / OUTPUT_CSV_NAME
    final_df.to_csv(output_csv, index=False, encoding="utf-8-sig")

    print(f"[DONE] CSV salvato in:\n{output_csv}")
    print(f"[DONE] Shape finale: {final_df.shape}")


if __name__ == "__main__":
    main()

[INFO] Base path: C:\Users\a.genua\OneDrive - Scuola Superiore Sant'Anna\PROGETTI\Progetti in corso\Articoli in corso\RA-L_&_Sanseverino\FlightSimulator
[INFO] Soggetti trovati: ['SUB1', 'SUB2', 'SUB3', 'SUB4', 'SUB5', 'SUB6', 'SUB7', 'SUB8', 'SUB9', 'SUB10', 'SUB11', 'SUB12', 'SUB13', 'SUB14', 'SUB15', 'SUB16', 'SUB17', 'SUB18', 'SUB19', 'SUB20', 'SUB21', 'SUB22', 'SUB23', 'SUB24', 'SUB25', 'SUB26', 'SUB27', 'SUB28', 'SUB29', 'SUB30']

=== RISPOSTE TROVATE NELL'EXCEL ===
('SUB1', 'HIGH') 28 ['Red', 'Green', 'Red', 'Green', 'Green']
('SUB1', 'LOW') 26 ['Blue', 'Yellow', 'Red', 'Red', 'Blue']
('SUB1', 'NOMINAL') 27 ['Red', 'Red', 'Yellow', 'Red', 'Green']
('SUB10', 'HIGH') 26 ['Blue', 'Yellow', 'Blue', 'Yellow', 'Blue']
('SUB10', 'LOW') 27 ['Green', 'Yellow', 'Green', 'Blue', 'Red']
('SUB10', 'NOMINAL') 23 ['Blue', 'Blue', 'Yellow', 'Red', 'Blue']
('SUB11', 'HIGH') 28 ['Blue', 'Yellow', 'Blue', 'Blue', 'Blue']
('SUB11', 'LOW') 28 ['Blue', 'Red', 'Green', 'Yellow', 'Blue']
('SUB11', 'NOM